<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

// Товар
public class Item
{
    public string Name { get; set; }
    public int Amount { get; set; }
    public string Delivery { get; set; }
}

// Интерфейсы 
public interface ITrackable
{
    void Track();
    string GetTrackingInfo();
}

public interface ICancelable
{
    void CancelOrder(string reason);
    bool IsCancelable { get; }
}

public interface IReturnable
{
    void ReturnItem(Item item, string reason);
    List<string> ReturnInfo { get; }
}

// Базовый класс Order
public class Order
{
    public int OrderId { get; set; }
    public string CreationDate { get; set; } 
    public int TotalAmount { get; set; }

    // Доп. атрибуты
    public string Status { get; set; } = "New";
    public string Currency { get; set; } = "RUB";
    public bool IsPaid { get; set; } = false;
    public double DiscountPercent { get; set; } = 0.0;
    public string CreatedBy { get; set; } = "system";

    public List<string> Names = new List<string>();
    public List<int> Amounts = new List<int>();
    public List<string> Notes = new List<string>();

    public virtual void CalculateTotal()
    {
        int sum = 0;
        for (int i = 0; i < Amounts.Count; i++)
            sum += Amounts[i];

        double afterDiscount = sum * (1 - DiscountPercent / 100.0);
        TotalAmount = (int)Math.Round(afterDiscount);
        Console.WriteLine($"Сумма заказа (после скидки {DiscountPercent}%): {TotalAmount} {Currency}");
    }

    public virtual void AddItem(Item item)
    {
        Amounts.Add(item.Amount);
        Names.Add(item.Name);
        Console.WriteLine($"Добавлен товар '{item.Name}' ({item.Amount} {Currency})");
    }

    public virtual void RemoveItem(Item item)
    {
        int idx = Names.FindIndex(n => n == item.Name);
        if (idx >= 0 && idx < Amounts.Count)
        {
            Names.RemoveAt(idx);
            Amounts.RemoveAt(idx);
            Console.WriteLine($"Удален товар '{item.Name}' из заказа");
        }
        else
        {
            Console.WriteLine($"Товар '{item.Name}' не найден");
        }
    }

    public virtual void ApplyDiscount(double percent)
    {
        DiscountPercent = percent;
        Console.WriteLine($"Установлена скидка: {percent}%");
    }

   public virtual void PrintSummary()
    {
        Console.WriteLine("----- Сводка заказа -----");
        Console.WriteLine($"Номер заказа: {OrderId}, Дата: {CreationDate}, Статус: {Status}, Создан: {CreatedBy}");
        Console.WriteLine($"Товары ({Names.Count}): {string.Join(", ", Names)}");
        Console.WriteLine($"Итого: {TotalAmount} {Currency} (Скидка: {DiscountPercent}%");
        if (Notes.Any()) Console.WriteLine("Заметки: " + string.Join(" | ", Notes));
        Console.WriteLine("-------------------------");
    }


    public virtual bool ValidateOrder()
    {
        bool ok = Names.Count > 0;
        Console.WriteLine(ok ? "Заказ валиден" : " Заказ пустой");
        return ok;
    }
}

// Онлайн-заказ (Order -> OnlineOrder) + интерфейсы
public class OnlineOrder : Order, ITrackable, ICancelable
{
    public string CustomerEmail { get; set; }
    public string PaymentMethod { get; set; } = "Card";
    public string DeliveryProvider { get; set; } = "Postal";
    public string TrackingNumber { get; protected set; }
    public DateTime? EstimatedDelivery { get; set; }

    protected List<string> Deliveries = new List<string>();public override void AddItem(Item item)
    {
        base.AddItem(item);
        Deliveries.Add(item.Delivery ?? "Не указана");
        Console.WriteLine($"Добавлена информация о доставке: {item.Delivery}");
    }

    public void SendDigitalReceipt()
    {
        if (string.IsNullOrEmpty(CustomerEmail))
            Console.WriteLine("Нет email для отправки чека");
        else
            Console.WriteLine($"Отправлен цифровой чек на {CustomerEmail}");
    }

    public void UpdateTracking(string newTracking, DateTime eta)
    {
        TrackingNumber = newTracking;
        EstimatedDelivery = eta;
        Console.WriteLine($"Трек-номер обновлён: {TrackingNumber}, ETA: {EstimatedDelivery}");
    }

    public void Track()
    {
        Console.WriteLine($"Трек-номер: {TrackingNumber ?? "не назначен"}, провайдер: {DeliveryProvider}");
    }

    public string GetTrackingInfo()
    {
        return $"Provider: {DeliveryProvider}, Tracking: {TrackingNumber ?? "N/A"}";
    }

    public bool IsCancelable => Status != "Shipped" && Status != "Delivered";

    public void CancelOrder(string reason)
    {
        if (!IsCancelable)
        {
            Console.WriteLine("Заказ нельзя отменить (уже отправлен/доставлен)");
            return;
        }
        Status = "Cancelled";
        Console.WriteLine($"Заказ отменен. Причина: {reason}");
    }
}

// Физический заказ (Order -> PhysicalOrder) + возвраты
public class PhysicalOrder : Order, IReturnable, ICancelable
{
    public string DeliveryAddress { get; set; }
    public double WeightKg { get; set; } = 0.0;
    public double ShippingCost { get; set; } = 0.0;
    public string ReturnPolicy { get; set; } = "14 days";

    public List<string> ReturnInfo { get; } = new List<string>();

    public override void RemoveItem(Item item)
    {
        int idx = Names.FindIndex(n => n == item.Name);
        if (idx >= 0 && idx < Amounts.Count)
        {
            ReturnInfo.Add($"Возврат товара '{item.Name}' ({DateTime.Now:dd.MM.yyyy})");
            Names.RemoveAt(idx);
            Amounts.RemoveAt(idx);
            Console.WriteLine($" Удален товар '{item.Name}'. Информация о возврате добавлена.");
        }
        else
        {
            Console.WriteLine($"Товар '{item.Name}' не найден");
        }
    }

    public void SchedulePickup(DateTime pickupDate)
    {
        Console.WriteLine($"Запланирован забор товара на {pickupDate:dd.MM.yyyy HH:mm}");
    }

    public void CalculateShipping()
    {
        ShippingCost = Math.Max(100, WeightKg * 100);
        Console.WriteLine($"Рассчитана стоимость доставки: {ShippingCost} {Currency}");
    }

    public void ReturnItem(Item item, string reason)
    {
        ReturnInfo.Add($"Возврат '{item.Name}': {reason} ({DateTime.Now:dd.MM.yyyy})");
        Console.WriteLine($"Запрошен возврат товара '{item.Name}'. Причина: {reason}");
    }

    public bool IsCancelable => Status != "Shipped" && Status != "Delivered";

    public void CancelOrder(string reason)
    {
        if (!IsCancelable)
        {
            Console.WriteLine("Нельзя отменить — уже отправлен/доставлен");
            return;
        }
        Status = "Cancelled";
        Console.WriteLine($"Заказ отменен. Причина: {reason}");
    }

    public override void PrintSummary()
    {
        base.PrintSummary();
        Console.WriteLine($"Адрес: {DeliveryAddress}, Масса: {WeightKg}kg, Стоимость доставки: {ShippingCost} {Currency}");
        if (ReturnInfo.Any()) Console.WriteLine(string.Join("; ", ReturnInfo));
    }
}

 // Generic-класс: репозиторий заказов
public class OrderRepository<T> where T : Order
{
    private readonly List<T> _store = new List<T>();

    public void Add(T order)
    {
        _store.Add(order);
        Console.WriteLine($"[Repository] Добавлен заказ #{order.OrderId} ({typeof(T).Name})");
    }

    public IEnumerable<T> GetAll() => _store;

    public T GetById(int id) => _store.FirstOrDefault(o => o.OrderId == id);

    public bool RemoveById(int id)
    {
        var item = GetById(id);
        if (item == null) return false;
        _store.Remove(item);
        return true;
    }

    // Вызов полиморфных методов у всех заказов
    public void PrintAllSummaries()
    {
        foreach (var order in _store)
            {
                order.PrintSummary(); // вызовет переопределённую версию, если есть
            }
    }
}

// Создаём репозиторий для заказов (generic)
var repo = new OrderRepository<Order>();

// Базовый Order
Order order = new Order { OrderId = 1001, CreationDate = "03.09.2025", CreatedBy = "operator1"};
Item it1 = new Item { Name = "Тетрадь", Amount = 200 };
Item it2 = new Item { Name = "Портфель", Amount = 2300 };
order.AddItem(it1);
order.AddItem(it2);
order.ApplyDiscount(5);
order.CalculateTotal();
order.PrintSummary();
Console.WriteLine();

// OnlineOrder
OnlineOrder onOrder = new OnlineOrder
{
OrderId = 2002,
CreationDate = "04.09.2025",
CustomerEmail = "client@example.com",
DeliveryProvider = "Почта России",
};

Item o1 = new Item { Name = "Гуашь", Amount = 800, Delivery = "Стандарт" };
onOrder.AddItem(o1);
onOrder.UpdateTracking("RU123456789", DateTime.Now.AddDays(5));
onOrder.ApplyDiscount(10);
onOrder.CalculateTotal();
onOrder.SendDigitalReceipt();
onOrder.Track();
Console.WriteLine();

// PhysicalOrder
PhysicalOrder phOrder = new PhysicalOrder { OrderId = 3003, CreationDate = "05.09.2025", DeliveryAddress = "ул. Ленина, 1", WeightKg = 5.0 };
Item p1 = new Item { Name = "Книга", Amount = 1200 };
phOrder.AddItem(p1);
phOrder.CalculateShipping();
phOrder.CalculateTotal();
phOrder.ReturnItem(p1, "Скучная");
phOrder.CancelOrder("Скучная");
Console.WriteLine();

Добавлен товар 'Тетрадь' (200 RUB)
Добавлен товар 'Портфель' (2300 RUB)
Установлена скидка: 5%
Сумма заказа (после скидки 5%): 2375 RUB
----- Сводка заказа -----
Номер заказа: 1001, Дата: 03.09.2025, Статус: New, Создан: operator1
Товары (2): Тетрадь, Портфель
Итого: 2375 RUB (Скидка: 5%
-------------------------

Добавлен товар 'Гуашь' (800 RUB)
Добавлена информация о доставке: Стандарт
Трек-номер обновлён: RU123456789, ETA: 11/23/2025 1:33:38 PM
Установлена скидка: 10%
Сумма заказа (после скидки 10%): 720 RUB
Отправлен цифровой чек на client@example.com
Трек-номер: RU123456789, провайдер: Почта России

Добавлен товар 'Книга' (1200 RUB)
Рассчитана стоимость доставки: 500 RUB
Сумма заказа (после скидки 0%): 1200 RUB
Запрошен возврат товара 'Книга'. Причина: Скучная
Заказ отменен. Причина: Скучная

